# Disc Detection

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Needed to import from the enderscope library
# RUN ONLY ONCE

import os

os.chdir("..")

In [ ]:
%matplotlib widget

## Imports

In [ ]:
from pathlib import Path

from tqdm import tqdm
from rich.pretty import pprint

import numpy as np
import pandas as pd
import cv2

import panel as pn

import enderleaf.const COLOR_SPACES
from enderleaf.draw import image_grid
from enderleaf.tools import read_dataframe, write_dataframe
from enderleaf.image import (
    load_image,
    to_pil,
    canny,
    find_circles,
    filter_circles,
    crop_image,
    Rectangle,
    match_previous_rotation,
    get_circles
)
from enderleaf.draw import draw_circles

## Constants

In [ ]:
PATH_TO_DATA = Path(".").joinpath("output", "job_data", "Exp26DM14", "I2")
PATH_TO_IMAGES = Path(".").joinpath("output", "images", "Exp26DM14", "I2")
MAX_CIRCLES=3

## Functions

In [ ]:
def load(row):
    return load_image(PATH_TO_IMAGES.joinpath(row.file_name))

## Load Data

In [ ]:
df = pd.concat([read_dataframe(f) for f in PATH_TO_DATA.glob("*.csv")]).sort_values(
    ["plate", "row", "col"]
)
df

In [ ]:
pn.extension("ipywidgets")

In [ ]:
sel_color_space = pn.widgets.Select(
    name="Color space", options=list(COLOR_SPACES.keys()), value="hsv", width=100
)
sel_channel = pn.widgets.Select(
    name="Channel", options=COLOR_SPACES[sel_color_space.value], value="s", width=100
)
ii_factor = pn.widgets.IntInput(
    name="Resize factor", start=1, end=20, step=1, value=4, width=100
)
ii_median_blur = pn.widgets.IntInput(
    name="Median blur", start=1, end=21, step=2, value=7, width=100
)
ii_gaussian_blur = pn.widgets.IntInput(
    name="Gaussian blur", start=1, end=21, step=2, value=1, width=100
)
chk_normalize = pn.widgets.Checkbox(name="Normalize", value=True)
irs_threshold = pn.widgets.IntRangeSlider(
    name="Thresholds",
    start=0,
    end=255,
    step=1,
    value=(150, 255),
    width=200,
)
sel_plate = pn.widgets.Select(name="Plate", options=list(df.plate.unique()), width=100)
sel_row = pn.widgets.Select(name="Row", options=list(df.row.unique()), width=100)
sel_col = pn.widgets.Select(name="Col", options=list(df.col.unique()), width=100)

dt_accepted = pn.pane.DataFrame(sizing_mode="stretch_width")
dt_discarded_position = pn.pane.DataFrame(sizing_mode="stretch_width")
dt_discarded_accu = pn.pane.DataFrame(sizing_mode="stretch_width")

bt_random = pn.widgets.Button(name="Random Disc")


def on_random(event):
    row = df[["plate", "row", "col"]].drop_duplicates().sample(n=1).iloc[0]
    sel_plate.value, sel_row.value, sel_col.value = row.plate, row.row, row.col


bt_random.on_click(on_random)


def filter_df() -> pd.DataFrame:
    return df[
        (df.plate == sel_plate.value)
        & (df.col == sel_col.value)
        & (df.row == sel_row.value)
    ]


ds_index = pn.widgets.DiscreteSlider(
    name="Timestamp",
    options=filter_df().sort_values("date_time").date_time.to_list(),
    sizing_mode="stretch_width",
)

img_transformed = pn.pane.Image(sizing_mode="stretch_width")
img_edges = pn.pane.Image(sizing_mode="stretch_width")
img_circles = pn.pane.Image(sizing_mode="stretch_width")
img_cropped = pn.pane.Image(sizing_mode="stretch_width")


def get_data(plate, row, col, timestamp):
    return df[
        (df.plate == plate)
        & (df.col == col)
        & (df.row == row)
        & (df.date_time == timestamp)
    ].iloc[0]


def get_circles(
    row, channel, thresholds, factor, normalize, median_blur, gaussian_blur
):
    image = load(row)
    im_width = image.shape[1] // factor
    im_height = image.shape[0] // factor
    image = cv2.resize(image, (im_width, im_height))
    if normalize is True:
        image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)
    if median_blur > 1:
        image = cv2.medianBlur(image, ksize=median_blur)
    if gaussian_blur > 1:
        image = cv2.GaussianBlur(image, ksize=(gaussian_blur, gaussian_blur), sigmaX=0)
    img_transformed.object = to_pil(image)
    edges = canny(
        image=image,
        color_space=sel_color_space.value,
        channel=channel,
        min_thresholf=thresholds[0],
        max_threshold=thresholds[1],
    )
    img_edges.object = to_pil(edges)

    return filter_circles(
        find_circles(
            edges=edges,
            radii=np.arange(450 // factor, 550 // factor, 20 // factor),
            max_circles=MAX_CIRCLES,
        ),
        img_width=im_width,
        img_height=im_height,
    )


@pn.depends(sel_color_space.param.value, watch=True)
def on_color_space_changed(cs):
    sel_channel.options = COLOR_SPACES[cs]
    sel_channel.value = COLOR_SPACES[cs][0]


@pn.depends(sel_plate.param.value, watch=True)
def on_plate_changed(plate):
    sel_row.value = sel_row.options[0]
    sel_col.value = sel_col.options[0]
    ds_index.value = ds_index.options[0]


@pn.depends(sel_col.param.value, sel_row.param.value, watch=True)
def on_pos_changed(col, row):
    _df = filter_df().sort_values("date_time")
    ds_index.options = _df.date_time.to_list()
    ds_index.value = ds_index.options[0]


@pn.depends(
    *[
        w.param.value
        for w in [
            sel_channel,
            ds_index,
            irs_threshold,
            ii_factor,
            chk_normalize,
            ii_median_blur,
            ii_gaussian_blur,
        ]
    ],
    watch=True,
)
def on_channel_selected(
    channel, index, thresholds, factor, normalize, median_blur, gaussian_blur
):
    row = get_data(
        plate=sel_plate.value, row=sel_row.value, col=sel_col.value, timestamp=index
    )
    circles = get_circles(
        row=row,
        channel=channel,
        thresholds=thresholds,
        factor=factor,
        normalize=normalize,
        median_blur=median_blur,
        gaussian_blur=gaussian_blur,
    )

    dt_accepted.object = pd.DataFrame(
        circles["accepted"], columns=["accu", "cx", "cy", "r"]
    )
    dt_discarded_position.object = pd.DataFrame(
        circles["discarded_position"], columns=["accu", "cx", "cy", "r"]
    )
    dt_discarded_accu.object = pd.DataFrame(
        circles["discarded_accu"], columns=["accu", "cx", "cy", "r"]
    )

    out = load(row)
    im_width = out.shape[1] // factor
    im_height = out.shape[0] // factor
    out = cv2.resize(out, (im_width, im_height))
    for accu, cx, cy, r in circles["accepted"]:
        out = cv2.circle(out, (cx, cy), r, (0, 0, 0), 4)
    for accu, cx, cy, r in circles["discarded_position"]:
        out = cv2.circle(out, (cx, cy), r, (255, 0, 0), 2)
    for accu, cx, cy, r in circles["discarded_accu"]:
        out = cv2.circle(out, (cx, cy), r, (255, 0, 255), 2)
    img_circles.object = to_pil(out)

    out_crop = load(row)
    out_crop = cv2.resize(out_crop, (im_width, im_height))
    if len(circles["accepted"]) == 1:
        accu, cx, cy, r = circles["accepted"][0]
        out_crop = cv2.resize(
            crop_image(out_crop, Rectangle.from_circle((cx, cy, r + 16))),
            (im_width, im_width),
        )
        img_cropped.object = to_pil(out_crop)


on_channel_selected(
    sel_channel.value,
    ds_index.value,
    irs_threshold.value,
    ii_factor.value,
    chk_normalize.value,
    gaussian_blur=ii_gaussian_blur.value,
    median_blur=ii_median_blur.value,
)

pn.Column(
    pn.layout.FlexBox(
        sel_plate,
        sel_row,
        sel_col,
        bt_random,
        sel_color_space,
        sel_channel,
        chk_normalize,
        ii_median_blur,
        ii_gaussian_blur,
        ii_factor,
        irs_threshold,
    ),
    pn.Row(
        pn.layout.WidgetBox("### Accepted", dt_accepted),
        pn.layout.WidgetBox("### Discarded position", dt_discarded_position),
        pn.layout.WidgetBox("### Discarded accu", dt_discarded_accu),
    ),
    ds_index,
    # pn.Row(img_transformed, img_edges),
    # pn.Row(img_circles, img_cropped),
    pn.Row(img_transformed, img_edges, img_circles, img_cropped),
)

In [ ]:
row = get_data(plate=sel_plate.value, row=sel_row.value, col=sel_col.value, timestamp=ds_index.value)
data = get_circles(
    row,
    sel_channel.value,
    irs_threshold.value,
    ii_factor.value,
    chk_normalize.value,
    gaussian_blur=ii_gaussian_blur.value,
    median_blur=ii_median_blur.value,
)
data

In [ ]:
def get_row_df(row) -> pd.DataFrame:
    return pd.DataFrame(row).T.reset_index(drop=True)


def get_circle_df(circles: dict) -> pd.DataFrame:
    return pd.concat(
        [
            pd.DataFrame(circles[key], columns=["accu", "cx", "cy", "radius"])
            .assign(kind=key)
            .reset_index(drop=True)
            for key in circles.keys()
        ]
    )


def merge_row_circles(row):
    return get_circle_df(
        get_circles(
            row,
            sel_channel.value,
            irs_threshold.value,
            ii_factor.value,
            chk_normalize.value,
            gaussian_blur=ii_gaussian_blur.value,
            median_blur=ii_median_blur.value,
        )
    ).assign(**row)

In [ ]:
path_to_file = PATH_TO_DATA.parent.joinpath("circles").with_suffix(".csv")
if path_to_file.is_file() is False:
    df_circles = pd.concat([merge_row_circles(r) for i, r in tqdm(list(df.iterrows()))])
    write_dataframe(df_circles, path_to_file)
else:
    df_circles = read_dataframe(path_to_file)

In [ ]:
df_circles.kind.unique()

In [ ]:
sel_plate = pn.widgets.Select(name="Plate", options=list(df.plate.unique()), width=100)
sel_row = pn.widgets.Select(name="Row", options=list(df.row.unique()), width=100)
sel_col = pn.widgets.Select(name="Col", options=list(df.col.unique()), width=100)

DSPOPT_ACCEPTED = ("Accepted", "accepted")
DSPOPT_DISCARDED_POSITION = ("Discarded: Position", "discarded_position")
DSPOPT_DISCARDED_ACCU = ("Discarded: Accumulator", "discarded_accu")
cbg_display = pn.widgets.CheckBoxGroup(
    name="Display options",
    options=[
        DSPOPT_ACCEPTED[0],
        DSPOPT_DISCARDED_POSITION[0],
        DSPOPT_DISCARDED_ACCU[0],
    ],
    value=[DSPOPT_ACCEPTED[0], DSPOPT_DISCARDED_POSITION[0], DSPOPT_DISCARDED_ACCU[0]],
)

bt_random = pn.widgets.Button(name="Random Disc")


def on_random(event):
    row = df[["plate", "row", "col"]].drop_duplicates().sample(n=1).iloc[0]
    sel_plate.value = row.plate
    sel_row.value = row.row
    sel_col.value = row.col


bt_random.on_click(on_random)


def filter_df() -> pd.DataFrame:
    return df_circles[
        (df_circles.plate == sel_plate.value)
        & (df_circles.col == sel_col.value)
        & (df_circles.row == sel_row.value)
    ]


ds_index = pn.widgets.DiscreteSlider(
    name="Timestamp",
    options=filter_df().sort_values("date_time").date_time.to_list(),
    sizing_mode="stretch_width",
)

img_coarse = pn.pane.Image(sizing_mode="stretch_width", enable_streaming=True)
img_cropped = pn.pane.Image(sizing_mode="stretch_width")
img_rotated = pn.pane.Image(sizing_mode="stretch_width")


def get_leaf_ts_data(plate, row, col, timestamp):
    return df_circles[
        (df_circles.plate == plate)
        & (df_circles.col == col)
        & (df_circles.row == row)
        & (df_circles.date_time == timestamp)
    ]


def get_circles(data):
    return data


@pn.depends(sel_plate.param.value, watch=True)
def on_plate_changed(plate):
    sel_row.value = sel_row.options[0]
    sel_col.value = sel_col.options[0]
    ds_index.value = ds_index.options[0]


@pn.depends(sel_col.param.value, sel_row.param.value, watch=True)
def on_pos_changed(col, row):
    ds_index.options = list(filter_df().sort_values("date_time").date_time.unique())
    ds_index.value = ds_index.options[0]


@pn.depends(
    *[w.param.value for w in [ds_index, cbg_display]],
    watch=True,
)
def on_channel_selected(index, display_options):
    data = get_leaf_ts_data(
        plate=sel_plate.value, row=sel_row.value, col=sel_col.value, timestamp=index
    )
    circles = get_circles(data=data).iloc[:, 1:5]
    # return circles
    circles = {
        kind: circles[circles.kind == kind].drop("kind", axis=1).values
        for kind in circles.kind.unique()
    }
    # return circles

    out_coarse = load(data.iloc[0])
    im_width = out_coarse.shape[1] // 4
    im_height = out_coarse.shape[0] // 4
    out_coarse = cv2.resize(out_coarse, (im_width, im_height))
    out_crop = load(data.iloc[0])
    out_crop = cv2.resize(out_crop, (im_width, im_height))
    if DSPOPT_ACCEPTED[1] in circles:
        if len(circles[DSPOPT_ACCEPTED[1]]) == 1:
            cx, cy, r = circles[DSPOPT_ACCEPTED[1]][0]
            out_crop = cv2.resize(
                crop_image(out_crop, Rectangle.from_circle((cx, cy, r + 16))),
                (im_width, im_width),
            )
            img_cropped.object = to_pil(out_crop)
        else:
            img_cropped.object = None
        if DSPOPT_ACCEPTED[0] in display_options:
            for cx, cy, r in circles[DSPOPT_ACCEPTED[1]]:
                out_coarse = cv2.circle(out_coarse, (cx, cy), r, (0, 0, 0), 4)
    else:
        img_cropped.object = None
    if (
        DSPOPT_DISCARDED_POSITION[1] in circles
        and DSPOPT_DISCARDED_POSITION[0] in display_options
    ):
        for cx, cy, r in circles[DSPOPT_DISCARDED_POSITION[1]]:
            out_coarse = cv2.circle(out_coarse, (cx, cy), r, (255, 0, 0), 2)
    if (
        DSPOPT_DISCARDED_ACCU[1] in circles
        and DSPOPT_DISCARDED_ACCU[0] in display_options
    ):
        for cx, cy, r in circles[DSPOPT_DISCARDED_ACCU[1]]:
            out_coarse = cv2.circle(out_coarse, (cx, cy), r, (255, 0, 255), 2)
    img_coarse.object = to_pil(out_coarse)

    current_index = ds_index.options.index(index)
    if current_index < len(ds_index.options) - 1:
        succ_data = get_leaf_ts_data(
            plate=sel_plate.value,
            row=sel_row.value,
            col=sel_col.value,
            timestamp=ds_index.options[current_index + 1],
        )
        succ_circles = get_circles(data=succ_data).iloc[:, 1:5]
        succ_circles = {
            kind: succ_circles[succ_circles.kind == kind].drop("kind", axis=1).values
            for kind in succ_circles.kind.unique()
        }
        # return succ_circles
        if (
            DSPOPT_ACCEPTED[1] in succ_circles
            and len(succ_circles[DSPOPT_ACCEPTED[1]]) == 1
        ):
            out_succ_crop = load(succ_data.iloc[0])
            out_succ_crop = cv2.resize(out_succ_crop, (im_width, im_height))
            cx, cy, r = circles[DSPOPT_ACCEPTED[1]][0]
            out_succ_crop = cv2.resize(
                crop_image(out_succ_crop, Rectangle.from_circle((cx, cy, r + 16))),
                (im_width, im_width),
            )
            img_rotated.object = to_pil(
                match_previous_rotation(
                    previous_image=out_crop, current_image=out_succ_crop
                )
            )


on_channel_selected(ds_index.value, cbg_display.value)

pn.Column(
    pn.layout.FlexBox(sel_plate, sel_row, sel_col, cbg_display, bt_random),
    ds_index,
    pn.Row(img_coarse, img_cropped, img_rotated),
)

In [ ]:
9,D,3 Disp
3,F,9 Cotton

In [ ]:
get_data(plate=sel_plate.value, row=sel_row.value, col=sel_col.value, timestamp=ds_index.value)

In [ ]:
video_stream = pn.widgets.VideoStream()
video_stream.stream()